In [1]:
from astropy.io import fits
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize as opt
import scipy.stats as st
from scipy.interpolate import interp1d
from stis_extract import *

In [2]:
rootdir = '/Users/thepoetoftwilight/Documents/CUBS/Data/CONTACT/STIS/'

In [3]:
plt.style.use('/Users/thepoetoftwilight/Documents/CUBS/Code/science.mplstyle')

# Visit 38

In [4]:
qso_name = 'J0454-6116'

In [6]:
d_v38 = coadd_1dspec(qso_name, ['OF8G38010', 'OF8G38020', 'OF8G38030'], 4)

## Compare with Zhijie's stitch

In [9]:
fits_zq = fits.open(rootdir+'J0454-6116/J0454_visit38.fits')
wave_zq = fits_zq[1].data['wave']
flux_zq = fits_zq[1].data['flux']
err_zq = fits_zq[1].data['error']

flux_zq_interp = interp1d(wave_zq, flux_zq, fill_value='extrapolate')(d_v38['wave'])
err_zq_interp = np.sqrt(interp1d(wave_zq, err_zq**2, fill_value='extrapolate')(d_v38['wave']))

/var/folders/tj/vc_wjrpj36sf3zws4s7s770c0000gn/T/ipykernel_37184/1549617222.py:7: RuntimeWarning: invalid value encountered in sqrt
  err_zq_interp = np.sqrt(interp1d(wave_zq, err_zq**2, fill_value='extrapolate')(d_v38['wave']))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10,5.), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

axes[0].step(wave_zq, flux_zq, where='mid', color='red', alpha=1, lw=.2, label="Flux (Zhijie)")
axes[0].step(wave_zq, err_zq, where='mid', color='blue', alpha=1, lw=.2, label="Error (Zhijie)")

axes[0].step(wav_stitch_v38, flux_stitch_v38, where='mid', color='black', alpha=1, lw=.2, label="Flux (Suyash)")
axes[0].step(wav_stitch_v38, err_stitch_v38, where='mid', color='cyan', alpha=1, lw=.2, label="Error (Suyash)")

axes[0].set_ylabel(r'Flux (erg/s/cm${}^2$/Å)')

#axes[0].legend(loc='upper right')
axes[0].set_xlim(2000,2800)
axes[0].set_ylim(-1e-16,2.5e-14)
axes[0].legend()
axes[0].set_title('J045415, visit 38 co-add')

axes[1].step(wav_stitch_v38, (flux_stitch_v38-flux_zq_interp)/np.sqrt(err_stitch_v38**2+err_zq_interp**2), 
             alpha=1,lw=.2, color='black')

axes[1].set_ylabel(r'$\Delta f_\lambda / \sigma_f$')
axes[1].set_xlabel('Wavelength (Å)')
axes[1].set_ylim(-1.9, 1.9)
plt.subplots_adjust(wspace=0.2, hspace=0)

## Flux calculation

Compute photometry

In [ ]:
idx = wav_stitch_v38<2730

flux_lam_ref = 3631/(3.34*1e4*wav_stitch_v38[idx]**2)
dwav = wav_stitch_v38[idx][1:]-wav_stitch_v38[idx][:-1]
wav_int = .5*(wav_stitch_v38[idx][1:]+wav_stitch_v38[idx][:-1])
f_int = .5*(flux_stitch_v38[idx][1:]+flux_stitch_v38[idx][:-1])
f_ref_int = .5*(flux_lam_ref[1:]+flux_lam_ref[:-1])

In [ ]:
mag = -2.5*np.log10(np.nansum(wav_int*f_int*dwav)/np.nansum(wav_int*f_ref_int*dwav))

In [ ]:
mag

Convert back to flux using pivot wavelength

In [ ]:
for i in range(len(flux_stitch_v38)):
    
    if ~np.isnan(flux_stitch_v38[i]):
        w0 = wav_stitch_v38[i]
        break

In [ ]:
w1 = 2730

In [ ]:
wav_p = np.sqrt((w1-w0)/(w0**-1 - w1**-1))

In [ ]:
f_lam_nuv = (3631*10**(-mag/2.5))/(3.34e4*wav_p**2)

In [15]:
f_lam_nuv

5.871901602364419e-15

Convert GALEX magnitude to flux

In [16]:
m_galex = 16.16
wav_p_galex = 2267

In [17]:
f_lam_galex = (3631*10**(-m_galex/2.5))/(3.34e4*wav_p_galex**2)

In [18]:
f_lam_galex

7.267361132343471e-15

## Plotabs conversion

In [19]:
idx = (~np.isnan(flux_stitch_v38))&(~np.isnan(err_stitch_v38))
save_spec1d_fits(rootdir+'J045415/', 'J045415_es=4', wav_stitch_v38[idx], flux_stitch_v38[idx], err_stitch_v38[idx])